# Plot Underfitting Overfitting

============================
Underfitting vs. Overfitting
============================

This example demonstrates the problems of underfitting and overfitting and
how we can use linear regression with polynomial features to approximate
nonlinear functions. The plot shows the function that we want to approximate,
which is a part of the cosine function. In addition, the samples from the
real function and the approximations of different models are displayed. The
models have polynomial features of different degrees. We can see that a
linear function (polynomial with degree 1) is not sufficient to fit the
training samples. This is called **underfitting**. A polynomial of degree 4
approximates the true function almost perfectly. However, for higher degrees
the model will **overfit** the training data, i.e. it learns the noise of the
training data.
We evaluate quantitatively **overfitting** / **underfitting** by using
cross-validation. We calculate the mean squared error (MSE) on the validation
set, the higher, the less likely the model generalizes correctly from the
training data.

## Setup

Import required libraries:

In [ ]:
# Authors: The scikit-learn developers
# SPDX-License-Identifier: BSD-3-Clause

import matplotlib.pyplot as plt
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

**Mean Squared Error (MSE) Loss:** The go-to loss for **regression** — predicting continuous values:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Squaring the errors means large mistakes are penalized much more than small ones. This makes MSE sensitive to outliers. For outlier-robust regression, consider MAE (Mean Absolute Error) or Huber loss.

**Cross-Validation:** A more robust evaluation than a single train/test split. K-fold cross-validation:
1. Split data into $k$ equal folds
2. Train on $k-1$ folds, test on the remaining one
3. Repeat $k$ times, each fold serving as the test set once
4. Average the $k$ scores

This gives a more reliable performance estimate because every data point is used for both training and testing. Common choice: $k=5$ or $k=10$.

**Model Training (.fit()):** The `.fit()` method is where the model learns from data. It adjusts the model's internal parameters to minimize prediction errors on the training data.

For different model types, `.fit()` does different things:
- **Linear models**: Finds the best-fit line/plane (minimizes squared error)
- **Decision trees**: Recursively splits data to separate classes/values
- **Neural networks**: Runs gradient descent over many epochs
- **Transformers (StandardScaler, PCA)**: Computes statistics (mean, variance, components) from the training data

**Model Prediction (.predict()):** After training, `.predict()` applies the learned model to new (unseen) data to generate predictions.

- **Classification**: Returns predicted class labels
- **Regression**: Returns predicted continuous values

The quality of predictions depends on how well the model was trained and whether the new data resembles the training distribution.

**Linear Regression:** The simplest predictive model — fits a straight line (or hyperplane) through the data that minimizes the sum of squared errors:

$$\hat{y} = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b$$

Each coefficient $w_i$ tells you how much the prediction changes when feature $x_i$ increases by 1, holding other features constant. The intercept $b$ is the prediction when all features are zero.

**Scatter Plot:** Displays individual data points as dots on a 2D plane. Each point's position is determined by its x and y values. Color and size can encode additional dimensions.

**Use for:** Exploring relationships between two continuous variables, spotting clusters, identifying outliers, and visualizing model predictions vs. actual values.

**Line Plot:** Connects data points with lines — ideal for showing trends over a continuous variable (often time). In ML, line plots are commonly used for:
- Training/validation loss curves (to detect overfitting)
- Learning rate schedules
- Time series data

**Random Seed:** Setting a seed ensures **reproducibility** — the same random numbers are generated each time the code runs. This is critical in ML experiments because:
- Train/test splits will be the same
- Weight initialization will be identical
- Any randomized algorithm (dropout, data augmentation) will behave consistently

Without a fixed seed, your results would vary between runs, making it impossible to debug or compare experiments.

In [ ]:
def true_fun(X):
    return np.cos(1.5 * np.pi * X)


np.random.seed(0)

n_samples = 30
degrees = [1, 4, 15]

X = np.sort(np.random.rand(n_samples))
y = true_fun(X) + np.random.randn(n_samples) * 0.1

plt.figure(figsize=(14, 5))
for i in range(len(degrees)):
    ax = plt.subplot(1, len(degrees), i + 1)
    plt.setp(ax, xticks=(), yticks=())

    polynomial_features = PolynomialFeatures(degree=degrees[i], include_bias=False)
    linear_regression = LinearRegression()
    pipeline = Pipeline(
        [
            ("polynomial_features", polynomial_features),
            ("linear_regression", linear_regression),
        ]
    )
    pipeline.fit(X[:, np.newaxis], y)

    # Evaluate the models using crossvalidation
    scores = cross_val_score(
        pipeline, X[:, np.newaxis], y, scoring="neg_mean_squared_error", cv=10
    )

    X_test = np.linspace(0, 1, 100)
    plt.plot(X_test, pipeline.predict(X_test[:, np.newaxis]), label="Model")
    plt.plot(X_test, true_fun(X_test), label="True function")
    plt.scatter(X, y, edgecolor="b", s=20, label="Samples")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.xlim((0, 1))
    plt.ylim((-2, 2))
    plt.legend(loc="best")
    plt.title(
        "Degree {}\nMSE = {:.2e}(+/- {:.2e})".format(
            degrees[i], -scores.mean(), scores.std()
        )
    )
plt.show()